In [1]:
import rapidsegment as rs

In [2]:
rs.__version__

'1.1.5'

In [3]:
from rapidsegment import StrategicSegmentBuilder, StrategicSegmentScore, UniversalDataLoader


In [4]:
data = UniversalDataLoader(file_path = r"/workspaces/RapidSegment/student_burnout_dropout_dataset_2.csv").load()

In [5]:
import pyarrow as pa

# Dictionary comprehension to get counts for all columns
null_counts = {name: data[name].null_count for name in data.column_names}
print(null_counts)
# Output example: {'col1': 0, 'col2': 3, 'col3': 1}


{'Student_ID': 0, 'Age': 0, 'Gender': 0, 'Year_of_Study': 0, 'Department': 0, 'Residence_Type': 0, 'Attendance_Percent': 51, 'Study_Hours_Per_Day': 35, 'Previous_GPA': 35, 'Backlogs': 15, 'Sleep_Hours': 35, 'Screen_Time_Hours': 36, 'Exercise_Freq_Per_Week': 0, 'Social_Activity_Score': 40, 'Part_Time_Job': 0, 'Family_Income_Bracket': 0, 'Financial_Stress_Score': 43, 'Family_Support_Score': 44, 'Stress_Level': 49, 'Anxiety_Score': 39, 'Motivation_Score': 41, 'Peer_Pressure_Score': 41, 'Counseling_Access': 0, 'Burnout_Level': 0, 'Dropout_Risk': 0}


In [6]:
print(data.schema)


Student_ID: string
Age: double
Gender: string
Year_of_Study: double
Department: string
Residence_Type: string
Attendance_Percent: double
Study_Hours_Per_Day: double
Previous_GPA: double
Backlogs: double
Sleep_Hours: double
Screen_Time_Hours: double
Exercise_Freq_Per_Week: double
Social_Activity_Score: double
Part_Time_Job: string
Family_Income_Bracket: string
Financial_Stress_Score: double
Family_Support_Score: double
Stress_Level: double
Anxiety_Score: double
Motivation_Score: double
Peer_Pressure_Score: double
Counseling_Access: string
Burnout_Level: string
Dropout_Risk: string


In [7]:

import pyarrow.compute as pc


# 2. Define your replacements
condition = pc.equal(data["Dropout_Risk"], "Yes")
new_values = pc.if_else (condition, 1, 0)

# 3. Replace the old column with the new computed array in the immutable table
new_table = data.set_column(
    data.schema.get_field_index("Dropout_Risk"), 
    "Dropout_Risk", 
    new_values
)


In [8]:
grid_config = {
    "min_sample_size": [100, 50],
    "min_lift": [2.0, 1.2]
}


In [9]:
builder = StrategicSegmentBuilder(
    target="Dropout_Risk",
    min_sample_size=50,
    min_lift=1.2,
    min_events=5,
    top_n_vars=10,
    max_segments=10,
    max_feature_reuse=1,
    param_grid=None,
    enable_diversity=True,
    feature_groups=None,
    ignore_features=["Student_ID"], 
    enable_1way= True
)

In [10]:
segments_summary = builder.extract_segments(new_table)

2026-07-17 16:27:44,051 | INFO     | [builder.py:329] | DuckDB Configured: Threads=4/4, MemoryLimit=7GB
2026-07-17 16:27:44,083 | INFO     | [builder.py:363] | Iteration 1 | Remaining Volume: 800 | Base Rate: 54.62%


2026-07-17 16:27:45,269 | INFO     | [builder.py:514] | Feature Usage Tracker Update -> 'Family_Income_Bracket' used count = 1
2026-07-17 16:27:45,270 | INFO     | [builder.py:514] | Feature Usage Tracker Update -> 'Burnout_Level' used count = 1
2026-07-17 16:27:45,270 | INFO     | [builder.py:514] | Feature Usage Tracker Update -> 'Study_Hours_Per_Day' used count = 1
2026-07-17 16:27:45,270 | INFO     | [builder.py:530] | Segment 1 Captured (Size Floor: 50 | Lift Floor: 1.2): rows=51, events=49.0, lift=1.76
  Rule: Family_Income_Bracket=<ArrowStringArray>
['Lower-Middle']
Length: 1, dtype: str & Burnout_Level=<ArrowStringArray>
['High']
Length: 1, dtype: str & Study_Hours_Per_Day=[0.05, 3.45)
  SQL: Family_Income_Bracket IN ('Lower-Middle') AND Burnout_Level IN ('High') AND (Study_Hours_Per_Day >= 0.05 AND Study_Hours_Per_Day < 3.45)
2026-07-17 16:27:45,282 | INFO     | [builder.py:363] | Iteration 2 | Remaining Volume: 749 | Base Rate: 51.80%
2026-07-17 16:27:46,557 | WARNING  | [bui

In [11]:
builder.explain_feature_journey("Stress_Level")

AUDIT TRAIL FOR FEATURE: 'Stress_Level'

[Iteration 1]
  • Current dynamic IV   : 13.3429
  • Previous times used  : 0
  • Selection Status     : Eligible for Combination Search
  • Winner this round    : Family_Income_Bracket=<ArrowStringArray>
['Lower-Middle']
Length: 1, dtype: str & Burnout_Level=<ArrowStringArray>
['High']
Length: 1, dtype: str & Study_Hours_Per_Day=[0.05, 3.45) (Variables: ['Family_Income_Bracket', 'Burnout_Level', 'Study_Hours_Per_Day'])

[Iteration 2]
  • Current dynamic IV   : 12.1751
  • Previous times used  : 0
  • Selection Status     : Eligible for Combination Search


In [12]:
import pandas as pd
segments_df = pd.DataFrame(segments_summary)
print("\nExtracted Segment Profiles:")
segments_df[["segment_id", "count", "lift", "meta_applied_sample_size", "sql_filter"]]



Extracted Segment Profiles:


,segment_id,count,lift,meta_applied_sample_size,sql_filter
0,1,51,1.758873,50,Family_Income_Bracket IN ('Lower-Middle') AND ...


In [13]:
segments_df

,segment_id,rule_string,sql_filter,count,rate,lift,meta_applied_sample_size,meta_applied_min_lift
0,1,Family_Income_Bracket=<ArrowStringArray>\n['Lo...,Family_Income_Bracket IN ('Lower-Middle') AND ...,51,96.078431,1.758873,50,1.2


In [14]:
coverage_report = builder.evaluate_final_coverage(new_table)
print("\nCascading Portfolio Coverage Analysis:")
pd.DataFrame(coverage_report)



Cascading Portfolio Coverage Analysis:


,segment,total_count,target_events,response_rate,base_response_rate,capture_rate,lift
0,0,749,388.0,51.802403,54.625,93.625,0.948328
1,1,51,49.0,96.078431,54.625,6.375,1.758873


In [15]:
print("--- FULL SEGMENT RULES ---\n")

for index, row in pd.DataFrame(segments_df).iterrows():
    print(f"Segment ID: {row['segment_id']}")
    print(f"Raw Rule:   {row['rule_string']}")
    print(f"SQL Filter: {row['sql_filter']}")
    print("-" * 50)

--- FULL SEGMENT RULES ---

Segment ID: 1
Raw Rule:   Family_Income_Bracket=<ArrowStringArray>
['Lower-Middle']
Length: 1, dtype: str & Burnout_Level=<ArrowStringArray>
['High']
Length: 1, dtype: str & Study_Hours_Per_Day=[0.05, 3.45)
SQL Filter: Family_Income_Bracket IN ('Lower-Middle') AND Burnout_Level IN ('High') AND (Study_Hours_Per_Day >= 0.05 AND Study_Hours_Per_Day < 3.45)
--------------------------------------------------


In [16]:
import duckdb
con = duckdb.connect(database='temp_scoring.db', read_only=False)
con.execute("CREATE OR REPLACE TABLE scored_data AS SELECT * FROM new_table")

In [17]:
con.execute("""SELECT
  CASE
    WHEN Burnout_Level IN ('High') THEN 1
    WHEN Attendance_Percent < 70.40 THEN 2
    WHEN Family_Support_Score < 5.35 THEN 3
    WHEN Motivation_Score < 4.95 THEN 4
    WHEN Stress_Level >= 5.35 THEN 5
    WHEN Part_Time_Job IN ('No') THEN 6
    ELSE 0
  END AS SEG,
  COUNT(*) AS count,
  SUM(Dropout_Risk) AS events
FROM scored_data
GROUP BY ALL""").fetchdf()

,SEG,count,events
0,0,30,7.0
1,1,200,184.0
2,2,148,87.0
3,3,122,57.0
4,4,109,48.0
5,5,112,40.0
6,6,79,14.0


In [18]:
con.close()

In [19]:
import pandas as pd
df = pd.read_csv(r"/workspaces/RapidSegment/student_burnout_dropout_dataset_2.csv")

In [20]:
df.isna().sum()

Student_ID                 0
Age                        0
Gender                    30
Year_of_Study              0
Department                23
Residence_Type            17
Attendance_Percent        51
Study_Hours_Per_Day       35
Previous_GPA              35
Backlogs                  15
Sleep_Hours               35
Screen_Time_Hours         36
Exercise_Freq_Per_Week     0
Social_Activity_Score     40
Part_Time_Job             25
Family_Income_Bracket     13
Financial_Stress_Score    43
Family_Support_Score      44
Stress_Level              49
Anxiety_Score             39
Motivation_Score          41
Peer_Pressure_Score       41
Counseling_Access          0
Burnout_Level              0
Dropout_Risk               0
dtype: int64

In [21]:

import duckdb
df = duckdb.query("""

SELECT * FROM df
WHERE NOT
(Family_Income_Bracket IN ('Lower-Middle') AND Burnout_Level IN ('High') AND (Study_Hours_Per_Day >= 0.05 AND Study_Hours_Per_Day < 3.45))
""").to_df()

In [22]:
df.Attendance_Percent = df.Attendance_Percent.fillna(0)

In [23]:
x = df.Attendance_Percent.values
y = df.Dropout_Risk.apply(lambda x: 1 if x=='Yes' else 0).values

In [24]:
from optbinning import OptimalBinning
optb = OptimalBinning(name='Attendance_Percent', dtype="numerical", solver="cp")
optb.fit(x, y)

,name,'Attendance_Percent'
,dtype,'numerical'
,prebinning_method,'cart'
,solver,'cp'
,divergence,'iv'
,max_n_prebins,20
,min_prebin_size,0.05
,min_n_bins,None
,max_n_bins,None
,min_bin_size,None
,max_bin_size,None


In [25]:
transformed_bins = optb.transform(x, metric="bins").astype(str)

In [26]:
print(pd.Series(transformed_bins).value_counts(dropna=False))

[78.75, 97.55)    267
[52.45, 68.05)    146
[68.05, 73.55)     93
(-inf, 52.45)      90
[97.55, inf)       56
[73.55, 76.95)     48
[76.95, 78.75)     45
Name: count, dtype: int64


In [27]:
optb.status

'OPTIMAL'

In [28]:
optb.splits

array([52.45000076, 68.04999924, 73.54999924, 76.95000076, 78.75      ,
       97.54999924])

In [29]:
optb.binning_table.build()

,Bin,Count,Count (%),Non-event,Event,Event rate,WoE,IV,JS
0,"(-inf, 52.45)",90,0.120805,28,62,0.688889,-0.733165,0.061510,0.007521
1,"[52.45, 68.05)",146,0.195973,50,96,0.657534,-0.590561,0.065845,0.008113
2,"[68.05, 73.55)",93,0.124832,36,57,0.612903,-0.397768,0.019377,0.002406
3,"[73.55, 76.95)",48,0.064430,25,23,0.479167,0.145146,0.001358,0.000170
4,"[76.95, 78.75)",45,0.060403,33,12,0.266667,1.073366,0.064577,0.007706
5,"[78.75, 97.55)",267,0.358389,157,110,0.411985,0.41753,0.061980,0.007692
6,"[97.55, inf)",56,0.075168,32,24,0.428571,0.349447,0.009135,0.001136
7,Special,0,0.000000,0,0,0.000000,0.0,0.000000,0.000000
8,Missing,0,0.000000,0,0,0.000000,0.0,0.000000,0.000000
Totals,,745,1.000000,361,384,0.515436,,0.283782,0.034743


In [30]:
duckdb.query("""SELECT COUNT(*), SUM(CASE WHEN Dropout_Risk = 'Yes' THEN 1 ELSE 0 END) FROM df WHERE Attendance_Percent <52.5
              GROUP BY ALL""")

┌──────────────┬──────────────────────────────────────────────────────────────┐
│ count_star() │ sum(CASE  WHEN ((Dropout_Risk = 'Yes')) THEN (1) ELSE 0 END) │
│    int64     │                            int128                            │
├──────────────┼──────────────────────────────────────────────────────────────┤
│           90 │                                                           62 │
└──────────────┴──────────────────────────────────────────────────────────────┘